## Load Volatility Data

In [13]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit

# Script-friendly repo path
repo_root = Path('..')
sys.path.insert(0, str(repo_root))

# Load .env from repo/CWD/HOME
_loaded_env_paths: list[str] = []
for dotenv_path in [repo_root / ".env", Path.cwd() / ".env", Path.home() / ".env"]:
    try:
        if dotenv_path.exists():
            load_dotenv(dotenv_path=dotenv_path, override=False)
            _loaded_env_paths.append(str(dotenv_path))
    except Exception:
        pass


def _read_tiingo_api_key() -> str:
    for k in [
        "TIINGO_API",
    ]:
        v = os.environ.get(k)
        if v and isinstance(v, str) and v.strip():
            return v.strip()
    return ""


In [14]:
from alphaforge.data.context import DataContext
from alphaforge.features.dataset_spec import (
    UniverseSpec,
    TimeSpec,
    FeatureRequest,
    TargetRequest,
    JoinPolicy,
    MissingnessPolicy,
)
from volatility_forecast.pipeline import (
    build_default_ctx,
    build_vol_dataset,
    VolDatasetSpec,
    build_wide_dataset,
)

# Features/targets
from volatility_forecast.features.return_features import (
    LagLogReturnTemplate,
    LagAbsLogReturnTemplate,
    LagSquaredLogReturnTemplate,
)
from volatility_forecast.features.selector import select_variant_columns
from volatility_forecast.targets.squared_return import NextDaySquaredReturnTarget

# Models
from volatility_forecast.model.es_model import ESModel
from volatility_forecast.model.stes_model import STESModel

# Simulated data
from volatility_forecast.reporting.plots import (
    plot_gate_diagnostics,
    plot_forecast_panel,
    plot_event_paths,
    plot_bar_series,
    plot_gate_panel,
)
from volatility_forecast.sources.simulated_garch import SimulatedGARCHSource


In [15]:

ENTITY = "SIMULATED"
SOURCE = "simulated_garch"
TABLE = "market.ohlcv"
PRICE_COL = "close"
CALENDAR = "XNYS"
GRID = "B"
START = pd.Timestamp("2000-01-01", tz="UTC")
END = pd.Timestamp("2023-01-01", tz="UTC")
N_LAGS = 0

IS_INDEX = 500
OS_INDEX = 2000
N_RUNS = 100

# SPY study settings
SPY_TICKER = "SPY"
SPY_START = pd.Timestamp("2000-01-01", tz="UTC")
SPY_END = pd.Timestamp("2023-12-31", tz="UTC")
SPY_IS_INDEX = 200
SPY_OS_INDEX = 4000
SPY_N_INITS = 100


def _attach_sim_source(ctx: DataContext, run_seed: int) -> None:
    """Attach a SimulatedGARCHSource to the context."""
    ctx.sources[SOURCE] = SimulatedGARCHSource(
        n_periods=2500,
        random_state=run_seed,
        mu=0.0,
        omega=0.02,
        alpha=0.11,
        beta=0.87,
        eta=4.0,
        shock_prob=0.005,
        entity_id=ENTITY,
    )


def _build_sim_spec(lags: int) -> VolDatasetSpec:
    """Spec for simulated data (raw/abs/squared lags)."""
    features = (
        FeatureRequest(
            template=LagLogReturnTemplate(),
            params={
                "lags": lags,
                "source": SOURCE,
                "table": TABLE,
                "price_col": PRICE_COL,
            },
        ),
        FeatureRequest(
            template=LagAbsLogReturnTemplate(),
            params={
                "lags": lags,
                "source": SOURCE,
                "table": TABLE,
                "price_col": PRICE_COL,
            },
        ),
        FeatureRequest(
            template=LagSquaredLogReturnTemplate(),
            params={
                "lags": lags,
                "source": SOURCE,
                "table": TABLE,
                "price_col": PRICE_COL,
            },
        ),
    )

    target = TargetRequest(
        template=NextDaySquaredReturnTarget(),
        params={"source": SOURCE, "table": TABLE, "price_col": PRICE_COL, "scale": 1.0},
        horizon=1,
        name="y",
    )

    return VolDatasetSpec(
        universe=UniverseSpec(entities=[ENTITY]),
        time=TimeSpec(start=START, end=END, calendar=CALENDAR, grid=GRID, asof=None),
        features=features,
        target=target,
        join_policy=JoinPolicy(how="inner", sort_index=True),
        missingness=MissingnessPolicy(final_row_policy="drop_if_any_nan"),
    )


def _build_spy_spec(lags: int) -> VolDatasetSpec:
    """Spec for SPY via Tiingo (raw/abs/squared lags)."""
    features = (
        FeatureRequest(
            template=LagLogReturnTemplate(),
            params={
                "lags": lags,
                "source": "tiingo",
                "table": "market.ohlcv",
                "price_col": "close",
            },
        ),
        FeatureRequest(
            template=LagAbsLogReturnTemplate(),
            params={
                "lags": lags,
                "source": "tiingo",
                "table": "market.ohlcv",
                "price_col": "close",
            },
        ),
        FeatureRequest(
            template=LagSquaredLogReturnTemplate(),
            params={
                "lags": lags,
                "source": "tiingo",
                "table": "market.ohlcv",
                "price_col": "close",
            },
        ),
    )

    target = TargetRequest(
        template=NextDaySquaredReturnTarget(),
        params={
            "source": "tiingo",
            "table": "market.ohlcv",
            "price_col": "close",
            "scale": 1.0,
        },
        horizon=1,
        name="y",
    )

    return VolDatasetSpec(
        universe=UniverseSpec(entities=[SPY_TICKER]),
        time=TimeSpec(
            start=SPY_START, end=SPY_END, calendar=CALENDAR, grid=GRID, asof=None
        ),
        features=features,
        target=target,
        join_policy=JoinPolicy(how="inner", sort_index=True),
        missingness=MissingnessPolicy(final_row_policy="drop_if_any_nan"),
    )




In [39]:
spy_spec = _build_spy_spec(0)

In [40]:
ctx = build_default_ctx(tiingo_api_key=_read_tiingo_api_key())

In [41]:
X, y, r, _ = build_wide_dataset(ctx, spy_spec, entity_id=SPY_TICKER)

In [42]:
X[select_variant_columns(X, "STES_EAESE")]

,const,lag.logret.fc5d3612d9c7,lag.abslogret.1ad490bcb584,lag.sqlogret.c2cf3aa70c10
ts_utc,,,,
2000-01-04 16:00:00+00:00,1.0,-0.039891,0.039891,1.591318e-03
2000-01-05 16:00:00+00:00,1.0,0.001787,0.001787,3.194479e-06
2000-01-06 16:00:00+00:00,1.0,-0.016202,0.016202,2.625040e-04
2000-01-07 16:00:00+00:00,1.0,0.056452,0.056452,3.186871e-03
2000-01-10 16:00:00+00:00,1.0,0.003425,0.003425,1.172830e-05
...,...,...,...,...
2023-12-21 16:00:00+00:00,1.0,0.009437,0.009437,8.906151e-05
2023-12-22 16:00:00+00:00,1.0,0.002008,0.002008,4.030918e-06
2023-12-26 16:00:00+00:00,1.0,0.004214,0.004214,1.775474e-05


In [43]:
y.tail()

ts_utc
2023-12-21 16:00:00+00:00    4.030918e-06
2023-12-22 16:00:00+00:00    1.775474e-05
2023-12-26 16:00:00+00:00    3.263152e-06
2023-12-27 16:00:00+00:00    1.426386e-07
2023-12-28 16:00:00+00:00    8.405139e-06
Name: __y__, dtype: float64